In [43]:
# Import de bibliotecas
import pandas as pd

## EDA

In [44]:
# Leitura de arquivos
clima = pd.read_csv('../dados/clima_rio.csv')

In [ ]:
# Vendo como sao os dados
clima.head()

,bairro,lat,lon,data,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,rain_sum,windspeed_10m_max,relative_humidity_2m_max,relative_humidity_2m_min,et0_fao_evapotranspiration
0,Catete,-22.926,-43.1786,2020-01-01,32.2,23.8,27.8,0.0,0.0,19.0,91,53,5.94
1,Catete,-22.926,-43.1786,2020-01-02,29.8,24.1,26.5,1.6,1.6,15.1,95,64,3.19
2,Catete,-22.926,-43.1786,2020-01-03,27.3,22.9,24.9,2.7,2.7,18.8,97,73,3.80
3,Catete,-22.926,-43.1786,2020-01-04,27.1,22.8,24.6,1.8,1.8,12.4,97,73,2.97
4,Catete,-22.926,-43.1786,2020-01-05,26.6,23.5,24.7,4.4,4.4,16.2,96,77,3.85


In [ ]:
# Vendo o tipo dos dados 
clima.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 217689 entries, 0 to 217688
Data columns (total 13 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   bairro                      217689 non-null  object 
 1   lat                         217689 non-null  float64
 2   lon                         217689 non-null  float64
 3   data                        217689 non-null  object 
 4   temperature_2m_max          217689 non-null  float64
 5   temperature_2m_min          217689 non-null  float64
 6   temperature_2m_mean         217689 non-null  float64
 7   precipitation_sum           217689 non-null  float64
 8   rain_sum                    217689 non-null  float64
 9   windspeed_10m_max           217689 non-null  float64
 10  relative_humidity_2m_max    217689 non-null  int64  
 11  relative_humidity_2m_min    217689 non-null  int64  
 12  et0_fao_evapotranspiration  217689 non-null  float64
dtypes: float64(9),

In [ ]:
# Vendo se existem dados nulos
clima.isnull().sum()

bairro                        0
lat                           0
lon                           0
data                          0
temperature_2m_max            0
temperature_2m_min            0
temperature_2m_mean           0
precipitation_sum             0
rain_sum                      0
windspeed_10m_max             0
relative_humidity_2m_max      0
relative_humidity_2m_min      0
et0_fao_evapotranspiration    0
dtype: int64

## Criacao de um dataset otimizado

In [ ]:
# Convertendo para datetime e extraindo mes e ano para a criacao de um dataset otimizado
clima['data'] = pd.to_datetime(clima['data'])
clima['mes'] = clima['data'].dt.month
clima['ano'] = clima['data'].dt.year

In [ ]:
# Definindo as estacoes
def get_estacao(mes):
    if mes in [12, 1, 2]: return 'Verao'
    if mes in [3, 4, 5]: return 'Outono'
    if mes in [6, 7, 8]: return 'Inverno'
    return 'Primavera'

In [ ]:
# Criacao do dataset otimizado apenas com as informacoes mais relevantes pra ML
clima_otimizado = clima.groupby(['bairro', 'lat', 'lon', 'ano', 'mes']).agg({
    'temperature_2m_max': ['mean', 'max'],
    'precipitation_sum': 'sum',
    'relative_humidity_2m_min': 'mean'
}).reset_index()

In [55]:
# Ajustando nomes das colunas
clima_otimizado.columns = ['bairro', 'lat', 'lon', 'ano', 'mes', 
'temp_max_media', 'temp_max_abs', 'chuva_total', 'umidade_min_media']

In [ ]:
# Aplicando a estacao no dataset
clima_otimizado['estacao'] = clima_otimizado['mes'].apply(get_estacao)